# P2 — Judge sensitivity on Colab Local Runtime → SSH server (no Google Drive)

تشغيل على **Colab Local Runtime** الذي يصل إلى Jupyter على سيرفر SSH. جميع الملفات تحفظ على السيرفر، لا تحتاج `drive.mount` أو نقل بيانات إلى `/content`.

**نفس مشروع الملف المرجعي:** `~/P2/P2/input` للملفات الأصلية، `~/P2/P2/results/<RUN_ID>/GPU` للتنبؤات، `EXPORTS` للملفات المضغوطة، و`LOCAL_WORK` لملفات تفريغ الذاكرة.

1. افتحي Colab واتصلي بـ **Connect to local runtime** عبر SSH tunnel كما تفعلين مع الملف القديم. الـKernel نفسه يجب أن يعمل على السيرفر وفي بيئة `research-env`.
2. شغلي **SERVER SETTINGS** أولاً قبل استيراد Torch. اضبطي GPU UUID المخصص لكِ إن اختلف، ثم Restart kernel عند تغييره.
3. تأكدي أن الملفين `P2_FINAL_SENIOR_ADJUDICATED_GOLD_v1.0.xlsx` و`P2_CPU_Baselines_v1.0.xlsx` في `~/P2/P2/input/` مع مطابقة SHA-256.
4. شغلي الخلايا من الأعلى إلى الأسفل. شغلي `mistral_fp16_offload` ثم `qwen_fp16_offload` (يفضّل إعادة بدء kernel بينهما). التنبؤات تُحفظ **بعد كل حالة**؛ الاستكمال تلقائي باستخدام RUN_ID نفسه.
5. E1: تغيير رموز الإجابة دون ترتيبها الدلالي؛ E2: تغيير ترتيب أسطر الفئات مع الاحتفاظ بالرمز الدال على كل فئة. التحليل **ليس مقارنة عائلات أو معماريات**.

> لم يتم تنفيذ GPU عند إعداد هذه النسخة؛ الاستنتاجات العلمية تعتمد على نتائج التنفيذ الفعلي.


# P2 — Judge Answer-Code and Option-Order Sensitivity (Google Colab / GPU)

**Purpose (not an architecture or model-family comparison):** assess how stable the *semantic failure judgments* are under controlled changes in (E1) answer-code assignment, and (E2) display order of rubric options, separately for Qwen3-8B and Mistral-7B-Instruct-v0.3.

**Same as supplied P2 notebook:** frozen FINAL GOLD + OOF baselines; T1–T4; evidence condition C; FP16 unquantized model weights; original target definitions; closed-choice next-token log-probability scoring; Qwen system/user messages with thinking disabled; Mistral single user message. This is a NEW sensitivity study, not an architecture comparison, and is **not yet executed**.

### Required uploads
* `P2_FINAL_SENIOR_ADJUDICATED_GOLD_v1.0.xlsx` (sheet `FINAL_GOLD`)
* `P2_CPU_Baselines_v1.0.xlsx` (sheets `OOF_T1`, `OOF_T2`, `OOF_T3`, `OOF_T4`)

Default paths: Google Drive `MyDrive/P2/P2/inputs/` (editable below). Each target/variant writes an append-only CSV after **each individual case**, permitting resume. Downloadable ZIP contains `results/`, `reports/`, `manifest/`.

**Run order:** 1 Mount/install → 2 Verify data → 3 Inspect plans → 4 Run one judge → 5 Repeat for second judge → 6 Export. In a new Colab session, rerun cells 1–3 and then resume cell 4. Prefer high-VRAM Colab GPU (A100 or 40+ GiB); on T4, 7B/8B FP16 may require CPU/disk offload and is **not guaranteed** to fit. Offload can be slow; no quantization fallback is performed.


In [1]:
# SERVER SETTINGS — MUST RUN BEFORE ANY torch / transformers import
from pathlib import Path
import os, sys, platform

PROJECT = Path.home() / 'P2' / 'P2'
RUN_ID = 'SERVER_JUDGE_SENSITIVITY_20261009_A'
# Same full GPU UUID from P2_SERVER_LOCAL_CPU_GPU; change if your allocation changes.
ASSIGNED_GPU = 'GPU-1ea7d7f4-f80e-5950-75b2-388da9383402'
# ASSIGNED_GPU = ''  # to use the GPU visible to the current server process
if ASSIGNED_GPU:
    if 'torch' in sys.modules and os.environ.get('CUDA_VISIBLE_DEVICES') != ASSIGNED_GPU:
        raise RuntimeError('Restart kernel, then run SERVER SETTINGS first to change CUDA_VISIBLE_DEVICES.')
    os.environ['CUDA_VISIBLE_DEVICES'] = ASSIGNED_GPU
os.environ['HF_HOME'] = str(Path.home() / '.cache' / 'p2_hf_cache')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Host:', platform.node(), 'Interpreter:', sys.executable)
print('Project:', PROJECT)
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES','default'))
# Dependencies (optional, only if research-env needs them):
# %pip install 'transformers==4.57.6' 'huggingface_hub==0.36.2' 'accelerate>=1.0' safetensors pandas openpyxl 'scikit-learn>=1.4,<2' psutil


Host: wkstn Interpreter: /home/users/fatma/research-env/bin/python
Project: /home/users/fatma/P2/P2
CUDA_VISIBLE_DEVICES: GPU-1ea7d7f4-f80e-5950-75b2-388da9383402


In [2]:
from pathlib import Path
import os, json, hashlib, gc, time, zipfile, shutil, platform, inspect, random
from itertools import permutations
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, cohen_kappa_score, balanced_accuracy_score
import torch, transformers
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM

ROOT = PROJECT
INPUT = PROJECT / 'input'  # same spelling and path as the supplied server notebook
RUN = PROJECT / 'results' / RUN_ID
GPU_OUTPUT = RUN / 'GPU'
RAW = GPU_OUTPUT / 'SENSITIVITY_PREDICTIONS'
REPORTS = GPU_OUTPUT / 'SENSITIVITY_REPORTS'
MANIFEST = GPU_OUTPUT / 'SENSITIVITY_MANIFEST'
EXPORTS = RUN / 'EXPORTS'
LOCAL_OFFLOAD = RUN / 'LOCAL_WORK' / 'p2_sensitivity_offload'
for x in (RAW, REPORTS, MANIFEST, EXPORTS, LOCAL_OFFLOAD): x.mkdir(parents=True,exist_ok=True)
assert INPUT.is_dir(), f'Missing input directory: {INPUT}'
assert RUN_ID and all(c.isalnum() or c in '_-' for c in RUN_ID)
# Server GPU memory budget: keep headroom; user may override for MIG/other allocations.
assert torch.cuda.is_available(), 'SSH server kernel has no available CUDA GPU; check allocation and restart.'
total_gpu_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
GPU_MAX_MEMORY_GIB = max(7, int(total_gpu_gib) - 6)
CPU_MAX_MEMORY_GIB = max(3, min(24, int(__import__('psutil').virtual_memory().available / 2**30) - 8))
MAX_LENGTH = 4096
TARGETS = ['T1','T2','T3','T4']
JUDGES = {
 'mistral_fp16_offload': {'model_id':'mistralai/Mistral-7B-Instruct-v0.3','revision':None},
 'qwen_fp16_offload': {'model_id':'Qwen/Qwen3-8B','revision':'e8bbd8252970581ea5b08b6a5b3e668adaf3161a'}
}
# Start with Mistral. Change to qwen_fp16_offload in a fresh session for the second judge.
JUDGE_TO_RUN = 'qwen_fp16_offload'#mistral_fp16_offload'
MAX_NEW_CASES = None  # set e.g. 80 to limit work per Colab session. None runs all missing cases.
SEED = 20261009
assert torch.cuda.is_available(), 'CUDA GPU required on connected SSH server.'
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
print('Input:', INPUT, 'GPU output:', GPU_OUTPUT, 'Exports:', EXPORTS)


/home/users/fatma/research-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition VRAM GiB: 94.97
Input: /home/users/fatma/P2/P2/input GPU output: /home/users/fatma/P2/P2/results/SERVER_JUDGE_SENSITIVITY_20261009_A/GPU Exports: /home/users/fatma/P2/P2/results/SERVER_JUDGE_SENSITIVITY_20261009_A/EXPORTS


In [5]:
for fn, expected in EXPECTED_SHA.items():
    p = INPUT / fn
    actual = sha256(p)

    print(f"\nFile: {fn}")
    print(f"Expected: {expected}")
    print(f"Actual:   {actual}")
    print(f"Match:    {actual == expected}")


File: P2_FINAL_SENIOR_ADJUDICATED_GOLD_v1.0.xlsx
Expected: ccc910b7bafbfa9607e93ef2eba8605f56f7ed87460ef74892d4877c7db4de65
Actual:   ccc910b7bafbfa9607e93ef2eba8605f56f7ed87460ef74892d4877c7db4de65
Match:    True

File: P2_CPU_Baselines_v1.0.xlsx
Expected: 4502e71bd6d1b6d2941b0b10650caaa433d7188bf958017d2fc646d328a31163
Actual:   4502e71bd6d1b6d2941b0b10650caaa433d7188bf958017d2fc646d328a31163
Match:    True


In [4]:

import hashlib
import pandas as pd

GOLD_NAME = 'P2_FINAL_SENIOR_ADJUDICATED_GOLD_v1.0.xlsx'
BASE_NAME = 'P2_CPU_Baselines_v1.0.xlsx'

EXPECTED_SHA = {
    GOLD_NAME: 'ccc910b7bafbfa9607e93ef2eba8605f56f7ed87460ef74892d4877c7db4de65',
    BASE_NAME: '4502e71bd6d1b6d2941b0b10650caaa433d7188bf958017d2fc646d328a31163'
}

EXPECTED_N = {
    'T1': 720,
    'T2': 474,
    'T3': 460,
    'T4': 474
}

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(2**20), b''):
            h.update(chunk)
    return h.hexdigest()


# Verify input files
for fn, expected in EXPECTED_SHA.items():
    p = INPUT / fn
    assert p.is_file(), f'Missing exact input: {p}'

    actual = sha256(p)
    print(fn, actual)

    assert actual == expected, (
        f'Input SHA-256 mismatch: {fn}\n'
        f'Expected: {expected}\n'
        f'Actual:   {actual}'
    )

print('SHA-256 verification PASSED')


# Load GOLD
gold = pd.read_excel(
    INPUT / GOLD_NAME,
    sheet_name='FINAL_GOLD'
).reset_index(drop=True)

assert (
    len(gold) == 720
    and gold.Case_ID.is_unique
    and gold.Prompt_ID.nunique() == 120
)

GOLD = gold.assign(
    Case_ID=gold.Case_ID.astype(str)
).set_index('Case_ID', drop=False)

print('GOLD validation PASSED')


# Load and verify baseline predictions
oof = {}

for t in TARGETS:
    d = pd.read_excel(
        INPUT / BASE_NAME,
        sheet_name='OOF_' + t
    )

    d['Case_ID'] = d.Case_ID.astype(str)

    assert (
        len(d) == EXPECTED_N[t]
        and d.Case_ID.is_unique
        and set(d.Case_ID).issubset(GOLD.index)
    ), f'{t}: Case validation failed'

    expected_labels = {1, 2, 3} if t == 'T4' else {0, 1}

    assert set(d.y_true.astype(int)) == expected_labels, (
        f'{t}: Label validation failed'
    )

    merged = d[['Case_ID', 'Prompt_ID']].merge(
        GOLD[['Case_ID', 'Prompt_ID']].reset_index(drop=True),
        on='Case_ID',
        suffixes=('_oof', '_gold'),
        validate='one_to_one'
    )

    assert len(merged) == len(d), (
        f'{t}: Case_ID merge mismatch'
    )

    assert (
        merged['Prompt_ID_oof'].astype(str)
        == merged['Prompt_ID_gold'].astype(str)
    ).all(), f'{t}: Prompt_ID mismatch'

    oof[t] = (
        d.sort_values('Case_ID', kind='stable')
         .reset_index(drop=True)
    )

print('OOF validation PASSED')
print({t: len(d) for t, d in oof.items()})


P2_FINAL_SENIOR_ADJUDICATED_GOLD_v1.0.xlsx ccc910b7bafbfa9607e93ef2eba8605f56f7ed87460ef74892d4877c7db4de65
P2_CPU_Baselines_v1.0.xlsx 4502e71bd6d1b6d2941b0b10650caaa433d7188bf958017d2fc646d328a31163
SHA-256 verification PASSED
GOLD validation PASSED
OOF validation PASSED
{'T1': 720, 'T2': 474, 'T3': 460, 'T4': 474}


In [6]:
'''GOLD_NAME='P2_FINAL_SENIOR_ADJUDICATED_GOLD_v1.0.xlsx'
BASE_NAME='P2_CPU_Baselines_v1.0.xlsx'
EXPECTED_SHA={GOLD_NAME:'ccc910b7bafbfa9607e93ef2eba8605f56f7ed87460ef74892d4877c7db4de65',
 BASE_NAME:'4502e71bd6d1b0b10650caaa433d7188bf958017d2fc646d328a31163'}
EXPECTED_N={'T1':720,'T2':474,'T3':460,'T4':474}
def sha256(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for chunk in iter(lambda:f.read(2**20),b''):h.update(chunk)
 return h.hexdigest()
for fn,expected in EXPECTED_SHA.items():
 p=INPUT/fn
 assert p.is_file(),f'Missing exact input: {p}'
 actual=sha256(p);print(fn,actual)
 assert actual==expected,'Input SHA-256 mismatch: wrong version. Do not bypass.'
gold=pd.read_excel(INPUT/GOLD_NAME,sheet_name='FINAL_GOLD').reset_index(drop=True)
assert len(gold)==720 and gold.Case_ID.is_unique and gold.Prompt_ID.nunique()==120
GOLD=gold.assign(Case_ID=gold.Case_ID.astype(str)).set_index('Case_ID',drop=False)
oof={}
for t in TARGETS:
 d=pd.read_excel(INPUT/BASE_NAME,sheet_name='OOF_'+t)
 d['Case_ID']=d.Case_ID.astype(str)
 assert len(d)==EXPECTED_N[t] and d.Case_ID.is_unique and set(d.Case_ID).issubset(GOLD.index)
 assert set(d.y_true.astype(int))==({1,2,3} if t=='T4' else {0,1})
 assert all(d[['Case_ID','Prompt_ID']].merge(gold[['Case_ID','Prompt_ID']],on='Case_ID',suffixes=('_oof','_gold')).eval('Prompt_ID_oof == Prompt_ID_gold'))
 oof[t]=d.sort_values('Case_ID',kind='stable').reset_index(drop=True)
print({t:len(d) for t,d in oof.items()})
'''

"GOLD_NAME='P2_FINAL_SENIOR_ADJUDICATED_GOLD_v1.0.xlsx'\nBASE_NAME='P2_CPU_Baselines_v1.0.xlsx'\nEXPECTED_SHA={GOLD_NAME:'ccc910b7bafbfa9607e93ef2eba8605f56f7ed87460ef74892d4877c7db4de65',\n BASE_NAME:'4502e71bd6d1b0b10650caaa433d7188bf958017d2fc646d328a31163'}\nEXPECTED_N={'T1':720,'T2':474,'T3':460,'T4':474}\ndef sha256(p):\n h=hashlib.sha256()\n with open(p,'rb') as f:\n  for chunk in iter(lambda:f.read(2**20),b''):h.update(chunk)\n return h.hexdigest()\nfor fn,expected in EXPECTED_SHA.items():\n p=INPUT/fn\n assert p.is_file(),f'Missing exact input: {p}'\n actual=sha256(p);print(fn,actual)\n assert actual==expected,'Input SHA-256 mismatch: wrong version. Do not bypass.'\ngold=pd.read_excel(INPUT/GOLD_NAME,sheet_name='FINAL_GOLD').reset_index(drop=True)\nassert len(gold)==720 and gold.Case_ID.is_unique and gold.Prompt_ID.nunique()==120\nGOLD=gold.assign(Case_ID=gold.Case_ID.astype(str)).set_index('Case_ID',drop=False)\noof={}\nfor t in TARGETS:\n d=pd.read_excel(INPUT/BASE_NAME,shee

In [7]:
RUBRICS = {'T1': {'codes': ['A', 'B'], 'map': {'A': 0, 'B': 1}, 'rubric': 'Decide only whether the response contains an identifiable semantic-coherence or authorised-grounding failure.\nA = no_failure: no identifiable failure under the supplied evidence.\nB = failure: at least one identifiable semantic-coherence or authorised-grounding failure.\nDo not classify the subtype. Choose only A or B.'}, 'T2': {'codes': ['A', 'B'], 'map': {'A': 0, 'B': 1}, 'rubric': 'The case is already known to contain a failure. Decide whether authorised grounding is involved.\nA = semantic_only: the failure is meaning-level but does not require unsupported/fabricated content or conflict with an authorised visible anchor.\nB = grounding_involved: the failure includes overlap with unsupported/fabricated content or authorised-anchor conflict, including hallucination-only cases.\nChoose only A or B.'}, 'T3': {'codes': ['A', 'B'], 'map': {'A': 0, 'B': 1}, 'rubric': 'The case belongs to one of two mechanism families.\nA = Core_SCF: topic/QUD shift, causal-chain failure, response-goal misalignment, discourse-structure failure, local-to-global breakdown, referential instability, or term-sense instability.\nB = OVR: self-contradiction, visible-context inconsistency, entity/role/attribute misattribution, temporal/world-state inconsistency, reasoning-answer mismatch, or mixed-context synthesis failure.\nChoose only A or B.'}, 'T4': {'codes': ['A', 'B', 'C'], 'map': {'A': 1, 'B': 2, 'C': 3}, 'rubric': 'The case is already known to contain a failure. Judge functional severity, not confidence.\nA = severity 1 (Minor): localized disruption with limited effect on the main answer.\nB = severity 2 (Moderate): materially weakens an important task-relevant relation or component.\nC = severity 3 (Severe): undermines the main answer, reverses meaning, contradicts the prompt/source, or makes the answer unreliable for the task.\nChoose only A, B, or C.'}}


In [8]:
# Stable, pre-registered deterministic variants; E1 and E2 DO NOT CHANGE TOGETHER.
# E1: order stays canonical, rotate the category-to-answer-code mapping.
# E2: category-to-answer-code mapping stays canonical, rotate visible option line order.
# T1/T2/T3: 2 mappings + 2 orders (the identity variants are intentionally shared).
# T4: 3 rotations for each experiment (rotations only, not all 6 permutations).

def variants(target):
 codes=RUBRICS[target]['codes']; n=len(codes)
 canonical_labels=[RUBRICS[target]['map'][c] for c in codes]
 out=[]
 for shift in range(n):
  out.append({'experiment':'E1_CODE','variant':f'code_rot{shift}',
    'label_to_code':{str(l):codes[(i+shift)%n] for i,l in enumerate(canonical_labels)},
    'display_order':canonical_labels[:]})
 for shift in range(1,n):
  out.append({'experiment':'E2_ORDER','variant':f'order_rot{shift}',
    'label_to_code':{str(l):codes[i] for i,l in enumerate(canonical_labels)},
    'display_order':canonical_labels[shift:]+canonical_labels[:shift]})
 return out
for t in TARGETS:print(t,variants(t))

SYSTEM = 'You are a conservative semantic-coherence evaluator.\nApply only the supplied target rubric to the visible text.\nDo not infer hidden facts. Do not output an explanation.\nYour decision space is closed: select exactly one of the permitted answer codes.'

def serialise_C(g):
 prompt=str(g['Prompt_Text'] if pd.notna(g['Prompt_Text']) else '').strip()
 source=str(g['Source_Context'] if pd.notna(g['Source_Context']) else '').strip()
 output=str(g['Model_Output'] if pd.notna(g['Model_Output']) else '').strip()
 if str(g['Source_Available']).lower()=='yes' and source:
  return f'Task:\n{prompt}\n\nAuthorised source/context:\n{source}\n\nResponse:\n{output}'
 return f'Task:\n{prompt}\n\nResponse:\n{output}'

# Semantic descriptions are taken directly from the supplied original rubrics.
LABEL_TEXT={
 'T1':{0:'no_failure: no identifiable failure under the supplied evidence.',1:'failure: at least one identifiable semantic-coherence or authorised-grounding failure.'},
 'T2':{0:'semantic_only: the failure is meaning-level but does not require unsupported/fabricated content or conflict with an authorised visible anchor.',1:'grounding_involved: the failure includes overlap with unsupported/fabricated content or authorised-anchor conflict, including hallucination-only cases.'},
 'T3':{0:'Core_SCF: topic/QUD shift, causal-chain failure, response-goal misalignment, discourse-structure failure, local-to-global breakdown, referential instability, or term-sense instability.',1:'OVR: self-contradiction, visible-context inconsistency, entity/role/attribute misattribution, temporal/world-state inconsistency, reasoning-answer mismatch, or mixed-context synthesis failure.'},
 'T4':{1:'severity 1 (Minor): localized disruption with limited effect on the main answer.',2:'severity 2 (Moderate): materially weakens an important task-relevant relation or component.',3:'severity 3 (Severe): undermines the main answer, reverses meaning, contradicts the prompt/source, or makes the answer unreliable for the task.'}
}
PREAMBLE={
 'T1':'Decide only whether the response contains an identifiable semantic-coherence or authorised-grounding failure.',
 'T2':'The case is already known to contain a failure. Decide whether authorised grounding is involved.',
 'T3':'The case belongs to one of two mechanism families.',
 'T4':'The case is already known to contain a failure. Judge functional severity, not confidence.'
}
TAIL={
 'T1':'Do not classify the subtype.',
 'T2':'',
 'T3':'',
 'T4':''
}
def rubric_variant(target,var):
 lines=[PREAMBLE[target]]
 for label in var['display_order']:
  lines.append(f"{var['label_to_code'][str(label)]} = {LABEL_TEXT[target][label]}")
 if TAIL[target]:lines.append(TAIL[target])
 allowed=RUBRICS[target]['codes']
 lines.append('Choose only '+(' or '.join(allowed) if len(allowed)==2 else ', '.join(allowed[:-1])+', or '+allowed[-1])+'.')
 return '\n'.join(lines)

def judge_chat(g,target,var,tokenizer,judge_name):
 user=rubric_variant(target,var)+'\n\nVisible case:\n'+serialise_C(g)
 if judge_name.startswith('qwen_'):
  messages=[{'role':'system','content':SYSTEM},{'role':'user','content':user}]
 else:messages=[{'role':'user','content':SYSTEM+'\n\n'+user}]
 kw={'tokenize':False,'add_generation_prompt':True}
 if judge_name.startswith('qwen_'):kw['enable_thinking']=False
 return tokenizer.apply_chat_template(messages,**kw)


T1 [{'experiment': 'E1_CODE', 'variant': 'code_rot0', 'label_to_code': {'0': 'A', '1': 'B'}, 'display_order': [0, 1]}, {'experiment': 'E1_CODE', 'variant': 'code_rot1', 'label_to_code': {'0': 'B', '1': 'A'}, 'display_order': [0, 1]}, {'experiment': 'E2_ORDER', 'variant': 'order_rot1', 'label_to_code': {'0': 'A', '1': 'B'}, 'display_order': [1, 0]}]
T2 [{'experiment': 'E1_CODE', 'variant': 'code_rot0', 'label_to_code': {'0': 'A', '1': 'B'}, 'display_order': [0, 1]}, {'experiment': 'E1_CODE', 'variant': 'code_rot1', 'label_to_code': {'0': 'B', '1': 'A'}, 'display_order': [0, 1]}, {'experiment': 'E2_ORDER', 'variant': 'order_rot1', 'label_to_code': {'0': 'A', '1': 'B'}, 'display_order': [1, 0]}]
T3 [{'experiment': 'E1_CODE', 'variant': 'code_rot0', 'label_to_code': {'0': 'A', '1': 'B'}, 'display_order': [0, 1]}, {'experiment': 'E1_CODE', 'variant': 'code_rot1', 'label_to_code': {'0': 'B', '1': 'A'}, 'display_order': [0, 1]}, {'experiment': 'E2_ORDER', 'variant': 'order_rot1', 'label_to_co

## 4. Execute judge — run separately for each model

Set `JUDGE_TO_RUN` above. Each variant is resumable from CSV; completed rows are never regenerated. Strict checks guard the input versions, model revision and exact experimental configuration. If Colab disconnects, reconnect to the SSH-backed local runtime and rerun with the same RUN_ID. **GPU session not run during notebook creation.**

In [9]:
def atomic_write_json(obj,path):
 tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False));os.replace(tmp,path)
def atomic_csv(df,path):
 tmp=path.with_suffix('.tmp');df.to_csv(tmp,index=False);os.replace(tmp,path)

def load_judge():
 assert JUDGE_TO_RUN in JUDGES
 cfg=JUDGES[JUDGE_TO_RUN].copy()
 rev=cfg['revision'] or HfApi().model_info(cfg['model_id']).sha
 assert len(rev)==40
 cfg['revision']=rev
 out=RAW/JUDGE_TO_RUN;out.mkdir(parents=True,exist_ok=True)
 pin=MANIFEST/f'{JUDGE_TO_RUN}_pinned.json'
 pin_contents={'model_id':cfg['model_id'],'revision':rev,'dtype':'float16','quantization':'none',
   'condition':'C','max_length':MAX_LENGTH,'gold_sha256':EXPECTED_SHA[GOLD_NAME],
   'base_sha256':EXPECTED_SHA[BASE_NAME],'seed':SEED,'transformers':transformers.__version__}
 if pin.exists():
  old=json.loads(pin.read_text())
  for key in ['model_id','revision','dtype','quantization','condition','max_length','gold_sha256','base_sha256','seed']:
   assert old[key]==pin_contents[key], f'Configuration changed: {key}'
 else:atomic_write_json(pin_contents,pin)
 tokenizer=AutoTokenizer.from_pretrained(cfg['model_id'],revision=rev)
 tokenizer.padding_side='left'
 if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
 code_ids={}
 for c in 'ABC':
  ids=tokenizer(c,add_special_tokens=False)['input_ids']
  assert len(ids)==1, f'Answer code {c} is NOT single-token: {ids}'
  code_ids[c]=ids[0]
 free,total=torch.cuda.mem_get_info()
 print('Free VRAM GiB',round(free/2**30,2), 'Server offload free GiB',round(shutil.disk_usage(str(LOCAL_OFFLOAD)).free/2**30,2))
 if free/2**30 < 7:raise RuntimeError('Insufficient GPU memory on SSH server; check assigned GPU.')
 if shutil.disk_usage(str(LOCAL_OFFLOAD)).free/2**30 < 20:raise RuntimeError('Insufficient server disk for model/offload.')
 model=AutoModelForCausalLM.from_pretrained(cfg['model_id'],revision=rev,torch_dtype=torch.float16,
      device_map='auto',max_memory={0:f'{GPU_MAX_MEMORY_GIB}GiB','cpu':f'{CPU_MAX_MEMORY_GIB}GiB'},
      low_cpu_mem_usage=True,offload_folder=str(LOCAL_OFFLOAD/JUDGE_TO_RUN),
      offload_state_dict=True,offload_buffers=True,attn_implementation='sdpa')
 assert not getattr(model,'is_loaded_in_4bit',False) and not getattr(model,'is_loaded_in_8bit',False)
 model.eval(); cfg['hf_device_map']={k:str(v) for k,v in model.hf_device_map.items()}
 atomic_write_json({**pin_contents,'device_map':cfg['hf_device_map'],'answer_code_token_ids':code_ids},MANIFEST/f'{JUDGE_TO_RUN}_loaded.json')
 return cfg,tokenizer,model,code_ids

@torch.inference_mode()
def predict_one(model,tokenizer,code_ids,prompt,allowed_codes):
 tokens=tokenizer(prompt,return_tensors='pt',add_special_tokens=False)
 token_count=int(tokens['input_ids'].shape[1])
 # Stop instead of silently truncating; ensure prompt, labels, and scoring position are unchanged.
 if token_count > MAX_LENGTH:
  return None, None, token_count, 'too_long'
 dev=torch.device('cuda:0')
 inputs={k:v.to(dev) for k,v in tokens.items()}
 kw={'use_cache':False}
 if 'logits_to_keep' in inspect.signature(model.forward).parameters:kw['logits_to_keep']=1
 elif 'num_logits_to_keep' in inspect.signature(model.forward).parameters:kw['num_logits_to_keep']=1
 else:raise RuntimeError('Model does not support last-token logits; avoid full-vocab logits memory spike.')
 logits=model(**inputs,**kw).logits[:,-1,:].float()
 score=torch.log_softmax(logits,dim=-1)[0]
 scores={c:float(score[code_ids[c]].cpu()) for c in allowed_codes}
 pred_code=max(scores,key=scores.get)
 return pred_code,scores,token_count,'ok'

def execute_judge():
 cfg,tokenizer,model,code_ids=load_judge()
 new_cases=0
 try:
  for target in TARGETS:
   d=oof[target]
   for var in variants(target):
    name=f"{target}_{var['experiment']}_{var['variant']}"
    path=RAW/JUDGE_TO_RUN/(name+'.csv')
    identity={'judge':JUDGE_TO_RUN,'model_revision':cfg['revision'],'target':target,'variant':var,'condition':'C',
      'gold_hash':EXPECTED_SHA[GOLD_NAME],'base_hash':EXPECTED_SHA[BASE_NAME]}
    signature=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()
    old=pd.read_csv(path,dtype={'Case_ID':str}) if path.exists() else pd.DataFrame()
    if len(old):
     assert old.Case_ID.is_unique and set(old.Case_ID).issubset(set(d.Case_ID))
     assert set(old.signature.astype(str))=={signature},f'Config mismatch {path}'
    completed=set(old.Case_ID.astype(str)) if len(old) else set()
    print('WORK',JUDGE_TO_RUN,name,len(completed),'/',len(d),flush=True)
    for r in d.itertuples(index=False):
     cid=str(r.Case_ID)
     if cid in completed:continue
     if MAX_NEW_CASES is not None and new_cases>=MAX_NEW_CASES:
      print('PAUSED: session quota',new_cases);return
     prompt=judge_chat(GOLD.loc[cid],target,var,tokenizer,JUDGE_TO_RUN)
     pred_code,scores,n_tokens,status=predict_one(model,tokenizer,code_ids,prompt,RUBRICS[target]['codes'])
     inv={c:int(l) for l,c in var['label_to_code'].items()}
     pred=inv[pred_code] if status=='ok' else None
     record={'Case_ID':cid,'Prompt_ID':r.Prompt_ID,'Fold':int(r.Fold),'y_true':int(r.y_true),
        'pred':pred,'pred_code':pred_code,'status':status,'token_count':n_tokens,
        'scores':json.dumps(scores) if scores else '', 'signature':signature,
        'experiment':var['experiment'],'variant':var['variant'],'condition':'C',
        'model_revision':cfg['revision'],'model_id':cfg['model_id'],
        'input_sha256':hashlib.sha256(prompt.encode()).hexdigest()}
     old=pd.concat([old,pd.DataFrame([record])],ignore_index=True)
     atomic_csv(old,path)
     new_cases+=1
    print('SAVED',path,len(old),flush=True)
 finally:
  del model,tokenizer
  gc.collect();torch.cuda.empty_cache()
 print('FINISHED',JUDGE_TO_RUN,new_cases)

execute_judge()


`torch_dtype` is deprecated! Use `dtype` instead!


Free VRAM GiB 87.24 Server offload free GiB 223.78


Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████| 5/5 [00:08<00:00,  1.64s/it]

WORK qwen_fp16_offload T1_E1_CODE_code_rot0 0 / 720


SAVED /home/users/fatma/P2/P2/results/SERVER_JUDGE_SENSITIVITY_20261009_A/GPU/SENSITIVITY_PREDICTIONS/qwen_fp16_offload/T1_E1_CODE_code_rot0.csv 720
WORK qwen_fp16_offload T1_E1_CODE_code_rot1 0 / 720
SAVED /home/users/fatma/P2/P2/results/SERVER_JUDGE_SENSITIVITY_20261009_A/GPU/SENSITIVITY_PREDICTIONS/qwen_fp16_offload/T1_E1_CODE_code_rot1.csv 720
WORK qwen_fp16_offload T1_E2_ORDER_order_rot1 0 / 720
SAVED /home/users/fatma/P2/P2/results/SERVER_JUDGE_SENSITIVITY_20261009_A/GPU/SENSITIVITY_PREDICTIONS/qwen_fp16_offload/T1_E2_ORDER_order_rot1.csv 720
WORK qwen_fp16_offload T2_E1_CODE_code_rot0 0 / 474
SAVED /home/users/fatma/P2/P2/results/SERVER_JUDGE_SENSITIVITY_20261009_A/GPU/SENSITIVITY_PREDICTIONS/qwen_fp16_offload/T2_E1_CODE_code_rot0.csv 474
WORK qwen_fp16_offload T2_E1_CODE_code_rot1 0 / 474
SAVED /home/users/fatma/P2/P2/results/SERVER_JUDGE_SENSITIVITY_20261009_A/GPU/SENSITIVITY_PREDICTIONS/qwen_fp16_offload/T2_E1_CODE_code_rot1.csv 474
WORK qwen_fp16_offload T2_E2_ORDER_order_ro

## 5. Reports — per-class results, stability vs canonical mapping, manifest

Runs on CPU; no GPU model is loaded. The identity variant `E1_CODE/code_rot0` is the **newly rerun** canonical control, not a replacement for any archived published result. Comparisons to old saved predictions should be made separately only after verifying the same case IDs and protocol. Sample sizes and missing/truncated counts are explicit.

In [13]:
def build_reports():
 summaries=[];confusions=[];predictions=[]
 for judge in JUDGES:
  for target in TARGETS:
   control_path=RAW/judge/f'{target}_E1_CODE_code_rot0.csv'
   control=pd.read_csv(control_path,dtype={'Case_ID':str}) if control_path.exists() else None
   if control is not None:
    control=control[control.status.eq('ok')][['Case_ID','pred']].rename(columns={'pred':'baseline_pred'})
   for var in variants(target):
    p=RAW/judge/f"{target}_{var['experiment']}_{var['variant']}.csv"
    if not p.exists():continue
    df=pd.read_csv(p,dtype={'Case_ID':str})
    good=df[df.status.eq('ok')].copy()
    labels=[1,2,3] if target=='T4' else [0,1]
    if len(good):
     yt=good.y_true.astype(int);yp=good.pred.astype(int)
     a=accuracy_score(yt,yp);f1=f1_score(yt,yp,labels=labels,average='macro',zero_division=0)
     bal=balanced_accuracy_score(yt,yp)
     qwk=cohen_kappa_score(yt,yp,labels=labels,weights='quadratic') if target=='T4' else float('nan')
     cm=confusion_matrix(yt,yp,labels=labels)
     for i,label in enumerate(labels):
      tp=int(cm[i,i]);support=int(cm[i].sum());predn=int(cm[:,i].sum())
      confusions.append({'judge':judge,'target':target,'experiment':var['experiment'],'variant':var['variant'],
       'class_label':label,'support':support,'predicted_n':predn,'true_positive':tp,
       'recall':tp/support if support else None,'precision':tp/predn if predn else 0})
     stability=float('nan')
     if control is not None:
      joined=good[['Case_ID','pred']].merge(control,on='Case_ID',validate='one_to_one')
      stability=float((joined.pred.astype(int)==joined.baseline_pred.astype(int)).mean()) if len(joined) else float('nan')
    else:a=f1=bal=qwk=stability=float('nan')
    summaries.append({'judge':judge,'target':target,'experiment':var['experiment'],'variant':var['variant'],
      'expected_n':EXPECTED_N[target],'saved_n':len(df),'valid_n':len(good),'excluded_n':len(df)-len(good),
      'complete':len(df)==EXPECTED_N[target] and df.Case_ID.is_unique and len(good)==EXPECTED_N[target],
      'macro_f1':f1,'accuracy':a,'balanced_accuracy':bal,'qwk':qwk,'agreement_with_new_control':stability})
    predictions.append(good.assign(judge=judge,target=target,experiment=var['experiment'],variant=var['variant']))
 summary=pd.DataFrame(summaries);per_class=pd.DataFrame(confusions)
 summary.to_csv(REPORTS/'summary.csv',index=False)
 per_class.to_csv(REPORTS/'per_class.csv',index=False)
 if predictions:pd.concat(predictions,ignore_index=True).to_csv(REPORTS/'all_valid_predictions.csv',index=False)
 meta={'analysis':'Answer-code assignment E1 and option order E2 sensitivity; not architecture comparison',
   'input_hashes':EXPECTED_SHA,'n_expected':EXPECTED_N,'model_profiles':JUDGES,
   'all_complete':len(summary)==sum(len(variants(t)) for t in TARGETS)*len(JUDGES) and summary.complete.all(),
   'date_created_utc':pd.Timestamp.utcnow().isoformat(),'run_id':RUN_ID}
 atomic_write_json(meta,MANIFEST/'report_manifest.json')
 display(summary)
 return summary
summary=build_reports()


/tmp/ipykernel_805999/3125875676.py:43: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  'date_created_utc':pd.Timestamp.utcnow().isoformat(),'run_id':RUN_ID}


TypeError: Object of type bool is not JSON serializable

In [15]:
# Save reports and one ZIP on the SSH server. No drive.mount or google.colab.files.download.
summary = build_reports()
zip_path = EXPORTS / (RUN_ID + '_BUNDLE.zip')
tmp_zip = zip_path.with_suffix('.tmp')
with zipfile.ZipFile(tmp_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(GPU_OUTPUT.rglob('*')):
        if p.is_file(): z.write(p, arcname=str(p.relative_to(RUN)))
os.replace(tmp_zip, zip_path)
print('Server archive:', zip_path)
print('Size (MiB):', round(zip_path.stat().st_size / 2**20, 2))
print('SSH example from Windows CMD:')
print(f'scp -P 402 -i PATH_TO_PRIVATE_KEY fatma@kuber.org.uk:{zip_path} .')



P2 E1/E2 REPORT GENERATION
Summary CSV: SAVED
Per-class CSV: SAVED
Valid predictions CSV: SAVED
JSON Manifest: SAVED
------------------------------------------------------------
Expected variants: 28
Saved variants: 28
All complete: True


,judge,target,experiment,variant,expected_n,saved_n,valid_n,excluded_n,complete,macro_f1,accuracy,balanced_accuracy,qwk,agreement_with_new_control
0,mistral_fp16_offload,T1,E1_CODE,code_rot0,720,720,720,0,True,0.427608,0.456944,0.587553,NaN,1.000000
1,mistral_fp16_offload,T1,E1_CODE,code_rot1,720,720,720,0,True,0.623697,0.691667,0.619173,NaN,0.348611
2,mistral_fp16_offload,T1,E2_ORDER,order_rot1,720,720,720,0,True,0.502588,0.515278,0.628924,NaN,0.933333
3,mistral_fp16_offload,T2,E1_CODE,code_rot0,474,474,474,0,True,0.400707,0.626582,0.505049,NaN,1.000000
4,mistral_fp16_offload,T2,E1_CODE,code_rot1,474,474,474,0,True,0.572753,0.675105,0.589812,NaN,0.862869
5,mistral_fp16_offload,T2,E2_ORDER,order_rot1,474,474,474,0,True,0.496878,0.641350,0.541509,NaN,0.921941
6,mistral_fp16_offload,T3,E1_CODE,code_rot0,460,460,460,0,True,0.485156,0.652174,0.496725,NaN,1.000000
7,mistral_fp16_offload,T3,E1_CODE,code_rot1,460,460,460,0,True,0.229708,0.284783,0.506006,NaN,0.150000
8,mistral_fp16_offload,T3,E2_ORDER,order_rot1,460,460,460,0,True,0.446822,0.719565,0.506739,NaN,0.867391
9,mistral_fp16_offload,T4,E1_CODE,code_rot0,474,474,474,0,True,0.441150,0.544304,0.450287,0.206302,1.000000


Server archive: /home/users/fatma/P2/P2/results/SERVER_JUDGE_SENSITIVITY_20261009_A/EXPORTS/SERVER_JUDGE_SENSITIVITY_20261009_A_BUNDLE.zip
Size (MiB): 2.03
SSH example from Windows CMD:
scp -P 402 -i PATH_TO_PRIVATE_KEY fatma@kuber.org.uk:/home/users/fatma/P2/P2/results/SERVER_JUDGE_SENSITIVITY_20261009_A/EXPORTS/SERVER_JUDGE_SENSITIVITY_20261009_A_BUNDLE.zip .


## Reporting rules / safeguards

* The two perturbations are **independent**: E1 rotates code-label assignments without moving lines; E2 rotates line positions without changing code-label assignments. These are rotations (not exhaustive permutations), chosen to bound GPU cost. In a two-label target there is one non-identity variant per experiment; in T4 there are two.
* The existing manuscript's FP16 judge results and the new control should be checked for matching revisions, prompt text, token context, and case inclusion before making direct numeric comparisons. New variants are sensitivity diagnostics **within each judge**, not relative performance rankings.
* Full completion requires both judges, every target, and every variant at full expected N with no oversized prompts. `complete=False` flags partial results. Excluded long cases are written with status, never silently omitted.
* Interpretation should emphasize **category recall, predicted-class frequency, macro-F1, ordinal QWK, and agreement with the new control**. A high agreement-to-control cannot alone establish validity. No finding is claimed until the cells have been run and outputs inspected.
* Input is not edited. Never overwrite GOLD or OOF sheets. No learning, training, or architecture comparison occurs.


In [14]:

# ============================================================
# P2 - E1/E2 REPORT GENERATION
# FIXED VERSION
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix
)


# ============================================================
# 1. FIX JSON SERIALIZATION
# ============================================================

def json_converter(obj):

    if isinstance(obj, (np.bool_,)):
        return bool(obj)

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()

    if isinstance(obj, Path):
        return str(obj)

    raise TypeError(
        f"Object of type {type(obj).__name__} "
        "is not JSON serializable"
    )


def atomic_write_json(obj, path):

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_suffix('.tmp')

    try:
        tmp.write_text(
            json.dumps(
                obj,
                indent=2,
                ensure_ascii=False,
                default=json_converter,
                allow_nan=False
            ),
            encoding='utf-8'
        )

        os.replace(tmp, path)

    finally:
        if tmp.exists():
            tmp.unlink()


# ============================================================
# 2. BUILD REPORTS
# ============================================================

def build_reports():

    summaries = []
    confusions = []
    predictions = []

    REPORTS.mkdir(parents=True, exist_ok=True)
    MANIFEST.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # Iterate over judges and targets
    # --------------------------------------------------------

    for judge in JUDGES:

        for target in TARGETS:

            # ------------------------------------------------
            # Load reference control predictions
            # ------------------------------------------------

            control_path = (
                RAW / judge /
                f'{target}_E1_CODE_code_rot0.csv'
            )

            control = (
                pd.read_csv(
                    control_path,
                    dtype={'Case_ID': str}
                )
                if control_path.exists()
                else None
            )

            if control is not None:

                control = (
                    control[
                        control.status.eq('ok')
                    ][['Case_ID', 'pred']]
                    .rename(
                        columns={
                            'pred': 'baseline_pred'
                        }
                    )
                )

            # ------------------------------------------------
            # Process all E1/E2 variants
            # ------------------------------------------------

            for var in variants(target):

                experiment = var['experiment']
                variant = var['variant']

                p = (
                    RAW / judge /
                    f"{target}_{experiment}_{variant}.csv"
                )

                if not p.exists():
                    continue

                df = pd.read_csv(
                    p,
                    dtype={'Case_ID': str}
                )

                good = df[
                    df.status.eq('ok')
                ].copy()

                labels = (
                    [1, 2, 3]
                    if target == 'T4'
                    else [0, 1]
                )

                # --------------------------------------------
                # Compute metrics
                # --------------------------------------------

                if len(good):

                    yt = good.y_true.astype(int)
                    yp = good.pred.astype(int)

                    a = accuracy_score(
                        yt, yp
                    )

                    f1 = f1_score(
                        yt,
                        yp,
                        labels=labels,
                        average='macro',
                        zero_division=0
                    )

                    bal = balanced_accuracy_score(
                        yt, yp
                    )

                    qwk = (
                        cohen_kappa_score(
                            yt,
                            yp,
                            labels=labels,
                            weights='quadratic'
                        )
                        if target == 'T4'
                        else float('nan')
                    )

                    cm = confusion_matrix(
                        yt,
                        yp,
                        labels=labels
                    )

                    # ----------------------------------------
                    # Per-class metrics
                    # ----------------------------------------

                    for i, label in enumerate(labels):

                        tp = int(cm[i, i])
                        support = int(cm[i].sum())
                        predn = int(cm[:, i].sum())

                        confusions.append({
                            'judge': judge,
                            'target': target,
                            'experiment': experiment,
                            'variant': variant,
                            'class_label': label,
                            'support': support,
                            'predicted_n': predn,
                            'true_positive': tp,
                            'recall': (
                                tp / support
                                if support else None
                            ),
                            'precision': (
                                tp / predn
                                if predn else 0
                            )
                        })

                    # ----------------------------------------
                    # Agreement with E1 control
                    # ----------------------------------------

                    stability = float('nan')

                    if control is not None:

                        joined = good[
                            ['Case_ID', 'pred']
                        ].merge(
                            control,
                            on='Case_ID',
                            validate='one_to_one'
                        )

                        if len(joined):

                            stability = float(
                                (
                                    joined.pred.astype(int)
                                    ==
                                    joined.baseline_pred.astype(int)
                                ).mean()
                            )

                else:

                    a = float('nan')
                    f1 = float('nan')
                    bal = float('nan')
                    qwk = float('nan')
                    stability = float('nan')

                # --------------------------------------------
                # Check experiment completeness
                # --------------------------------------------

                complete = bool(
                    len(df) == EXPECTED_N[target]
                    and df.Case_ID.is_unique
                    and len(good) == EXPECTED_N[target]
                )

                # --------------------------------------------
                # Store summary
                # --------------------------------------------

                summaries.append({
                    'judge': judge,
                    'target': target,
                    'experiment': experiment,
                    'variant': variant,
                    'expected_n': EXPECTED_N[target],
                    'saved_n': len(df),
                    'valid_n': len(good),
                    'excluded_n': len(df) - len(good),
                    'complete': complete,
                    'macro_f1': float(f1),
                    'accuracy': float(a),
                    'balanced_accuracy': float(bal),
                    'qwk': float(qwk),
                    'agreement_with_new_control': float(
                        stability
                    )
                })

                predictions.append(
                    good.assign(
                        judge=judge,
                        target=target,
                        experiment=experiment,
                        variant=variant
                    )
                )

    # ========================================================
    # 3. SAVE REPORT FILES
    # ========================================================

    summary = pd.DataFrame(summaries)
    per_class = pd.DataFrame(confusions)

    summary.to_csv(
        REPORTS / 'summary.csv',
        index=False
    )

    per_class.to_csv(
        REPORTS / 'per_class.csv',
        index=False
    )

    if predictions:

        pd.concat(
            predictions,
            ignore_index=True
        ).to_csv(
            REPORTS / 'all_valid_predictions.csv',
            index=False
        )

    # ========================================================
    # 4. CHECK COMPLETENESS
    # ========================================================

    expected_variants = (
        sum(
            len(variants(t))
            for t in TARGETS
        )
        * len(JUDGES)
    )

    all_complete = bool(
        len(summary) == expected_variants
        and bool(summary['complete'].all())
    )

    # ========================================================
    # 5. CREATE MANIFEST
    # ========================================================

    meta = {
        'analysis': (
            'Answer-code assignment E1 and '
            'option order E2 sensitivity; '
            'not architecture comparison'
        ),

        'input_hashes': EXPECTED_SHA,

        'n_expected': EXPECTED_N,

        'model_profiles': JUDGES,

        'all_complete': all_complete,

        'date_created_utc': (
            pd.Timestamp.now('UTC').isoformat()
        ),

        'run_id': RUN_ID
    }

    atomic_write_json(
        meta,
        MANIFEST / 'report_manifest.json'
    )

    # ========================================================
    # 6. DISPLAY RESULTS
    # ========================================================

    print("\n" + "=" * 60)
    print("P2 E1/E2 REPORT GENERATION")
    print("=" * 60)

    print("Summary CSV: SAVED")
    print("Per-class CSV: SAVED")

    if predictions:
        print("Valid predictions CSV: SAVED")

    print("JSON Manifest: SAVED")

    print("-" * 60)

    print(
        f"Expected variants: {expected_variants}"
    )

    print(
        f"Saved variants: {len(summary)}"
    )

    print(
        f"All complete: {all_complete}"
    )

    print("=" * 60)

    display(summary)

    return summary


# ============================================================
# 7. EXECUTE
# ============================================================

summary = build_reports()



P2 E1/E2 REPORT GENERATION
Summary CSV: SAVED
Per-class CSV: SAVED
Valid predictions CSV: SAVED
JSON Manifest: SAVED
------------------------------------------------------------
Expected variants: 28
Saved variants: 28
All complete: True


,judge,target,experiment,variant,expected_n,saved_n,valid_n,excluded_n,complete,macro_f1,accuracy,balanced_accuracy,qwk,agreement_with_new_control
0,mistral_fp16_offload,T1,E1_CODE,code_rot0,720,720,720,0,True,0.427608,0.456944,0.587553,NaN,1.000000
1,mistral_fp16_offload,T1,E1_CODE,code_rot1,720,720,720,0,True,0.623697,0.691667,0.619173,NaN,0.348611
2,mistral_fp16_offload,T1,E2_ORDER,order_rot1,720,720,720,0,True,0.502588,0.515278,0.628924,NaN,0.933333
3,mistral_fp16_offload,T2,E1_CODE,code_rot0,474,474,474,0,True,0.400707,0.626582,0.505049,NaN,1.000000
4,mistral_fp16_offload,T2,E1_CODE,code_rot1,474,474,474,0,True,0.572753,0.675105,0.589812,NaN,0.862869
5,mistral_fp16_offload,T2,E2_ORDER,order_rot1,474,474,474,0,True,0.496878,0.641350,0.541509,NaN,0.921941
6,mistral_fp16_offload,T3,E1_CODE,code_rot0,460,460,460,0,True,0.485156,0.652174,0.496725,NaN,1.000000
7,mistral_fp16_offload,T3,E1_CODE,code_rot1,460,460,460,0,True,0.229708,0.284783,0.506006,NaN,0.150000
8,mistral_fp16_offload,T3,E2_ORDER,order_rot1,460,460,460,0,True,0.446822,0.719565,0.506739,NaN,0.867391
9,mistral_fp16_offload,T4,E1_CODE,code_rot0,474,474,474,0,True,0.441150,0.544304,0.450287,0.206302,1.000000
